# Huấn luyện PCAP-Combined-Refit

Quy trình dùng split của [notebook tiền xử lý](data_preparation.ipynb). Các cell hiển thị dữ liệu và cấu hình học, sau đó đọc model đã fit. Đặt `FIT_AGAIN=True` để gọi cùng hàm huấn luyện với CLI, xuất sang thư mục mới. Chọn `prepared` theo kết quả của notebook trước nếu đã tạo split mới.

In [1]:
from datetime import datetime
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "configs/pcap_sessions_20261112.json").is_file())
sys.path.insert(0, str(root / "src"))
prepared = root / "experiments/pcap-scapy-20261112-seed42"
original = root / "experiments/pcap-combined-refit-seed42"

import numpy as np
from sklearn.preprocessing import LabelEncoder
from check_model import load_model
from train import CLASSES, MODEL_ID, SEED, SESSION_CAP, build_stack, prepare_refit, session_folds, train

FIT_AGAIN = False
run = original
if FIT_AGAIN:
    run = root / "experiments" / f"pcap-refit-{datetime.now():%Y%m%d-%H%M%S-%f}"
    train(prepared, run)
    print(f"Model và kết quả đã lưu tại {run}")

## 1. Chuẩn bị tập fit

Gộp train và validation của cả hai nguồn sau khi đã chốt cấu hình. Chọn tối đa 2.000 flow mỗi phiên với seed 42 để giảm ảnh hưởng của các phiên lớn. Sau khi chọn, khôi phục thứ tự dòng gốc trong phiên. Random Forest dùng bootstrap theo vị trí dòng nên cùng tập flow với thứ tự khác vẫn có thể tạo cây khác. Test được giữ riêng để chấm sau khi fit.

In [2]:
fit, tests = prepare_refit(prepared)
display(pd.crosstab(fit.source_dataset, fit.Label, margins=True))
display(fit.groupby(["source_dataset", "session_id"]).size().rename("Flow dùng để fit").to_frame())
print(f"Seed {SEED} · Cap {SESSION_CAP:,} · {len(fit):,} flow / {fit.session_id.nunique()} phiên")
print(f"Test gộp: {len(tests['combined']):,} flow / {tests['combined'].session_id.nunique()} phiên")

Label,BENIGN,BRUTE_FORCE,DOS,RECON,All
source_dataset,,,,,
cicids2017,4000,546,2933,3072,10551
insdn,3903,558,1295,4000,9756
All,7903,1104,4228,7072,20307


Flow dùng để fit
source_dataset session_id                          
cicids2017     cic-bfa-ftp                      316
               cic-bfa-ssh                      230
               cic-dos-hulk                    1000
               cic-dos-slowhttp                 933
               cic-dos-slowloris               1000
               cic-normal-a                    1000
               cic-normal-b                    1000
               cic-normal-c                    2000
               cic-recon-firewall                72
               cic-recon-sS                    1000
               cic-recon-sT                    2000
insdn          insdn-bfa-b                      411
               insdn-bfa-c                        1
               insdn-bfa-d                      146
               insdn-dos-a                      586
               insdn-dos-b                      660
               insdn-dos-c                       49
               insdn-normal-a                  1000
               insdn-normal-b                  2000
               insdn-normal-pilot               903
               insdn-recon-b                   1000
               insdn-recon-d                   2000
               insdn-recon-msf                 1000

Seed 42 · Cap 2,000 · 20,307 flow / 23 phiên
Test gộp: 26,475 flow / 8 phiên


## 2. Dự đoán ngoài fold theo phiên

Leave-One-Group-Out giữ riêng một phiên ở mỗi fold. Ba mô hình tầng 1 học từ các phiên còn lại rồi dự đoán xác suất cho phiên đang giữ. Mỗi flow nhận đúng một bộ dự đoán ngoài fold. Cách chia này giảm khả năng tầng meta học từ dự đoán được tạo bởi mô hình đã nhìn thấy flow của cùng phiên. Mỗi fold huấn luyện phải có đủ bốn lớp để giữ đúng thứ tự và số cột xác suất.

In [3]:
encoder = LabelEncoder().fit(CLASSES)
y = encoder.transform(fit.Label)
folds = session_folds(fit.session_id, y)
assert np.array_equal(np.sort(np.concatenate([test for _, test in folds])), np.arange(len(fit)))
for learn, held in folds:
    assert set(fit.session_id.iloc[learn]).isdisjoint(fit.session_id.iloc[held])
display(pd.DataFrame({
    "Phiên giữ riêng": [fit.session_id.iloc[held[0]] for _, held in folds],
    "Flow để học": [len(learn) for learn, _ in folds],
    "Flow nhận OOF": [len(held) for _, held in folds],
}))

,Phiên giữ riêng,Flow để học,Flow nhận OOF
0,cic-bfa-ftp,19991,316
1,cic-bfa-ssh,20077,230
2,cic-dos-hulk,19307,1000
3,cic-dos-slowhttp,19374,933
4,cic-dos-slowloris,19307,1000
5,cic-normal-a,19307,1000
6,cic-normal-b,19307,1000
7,cic-normal-c,18307,2000
8,cic-recon-firewall,20235,72
9,cic-recon-sS,19307,1000


## 3. Hai tầng của Stacking

Random Forest dùng bootstrap nhiều cây để giảm phương sai. XGBoost và LightGBM cộng dần các cây boosting để học quan hệ phi tuyến khác nhau. Median imputer nằm bên trong từng pipeline base để bước tiền xử lý chỉ được fit từ phần học của fold.

Tầng meta nhận 12 xác suất (3 mô hình × 4 lớp) và 12 đặc trưng flow. Nhánh flow dùng `log1p` để nén các biến lệch phải, sau đó chuẩn hóa để Logistic Regression xử lý các thang đo khác nhau. Xác suất đi thẳng vào meta. Logistic Regression học cách kết hợp đầu vào bằng mô hình tuyến tính có regularization. Sau khi tạo dữ liệu OOF, các base learner dùng để suy luận được fit trên toàn bộ tập phát triển.

In [4]:
recipe = build_stack(folds)
settings = ("n_estimators", "max_depth", "num_leaves", "learning_rate", "class_weight")
display(pd.DataFrame({name: {key: learner.steps[-1][1].get_params().get(key) for key in settings}
                      for name, learner in recipe.estimators}).T)
saved = load_model(run / f"{MODEL_ID}.joblib")["model"]
card = json.loads((run / "REFIT_CARD.json").read_text(encoding="utf-8"))
meta = saved.final_estimator_.steps[-1][1]
assert len(folds) == len(saved.cv) == card["fit_sessions"] and len(fit) == card["fit_rows"]
display(pd.Series({"Base learners": ", ".join(saved.named_estimators_),
                   "Meta-learner": type(meta).__name__, "C": meta.C,
                   "Số vòng tối đa": meta.max_iter,
                   "Số vòng hội tụ của bản đã lưu": int(meta.n_iter_[0]),
                   "Số đầu vào meta": meta.n_features_in_}).to_frame("Cấu hình"))

,n_estimators,max_depth,num_leaves,learning_rate,class_weight
rf,60,18,None,None,balanced_subsample
xgb,60.0,3.0,NaN,0.1,NaN
lgbm,60.0,10.0,15.0,0.08,NaN


,Cấu hình
Base learners,"rf, xgb, lgbm"
Meta-learner,LogisticRegression
C,0.1
Số vòng tối đa,3000
Số vòng hội tụ của bản đã lưu,74
Số đầu vào meta,24


Random Forest và hai thuật toán boosting dùng 60 cây hoặc vòng boosting. Báo cáo ghi thêm số fold và số vòng tối ưu của Logistic Regression. [Notebook đánh giá](evaluation.ipynb) nạp artifact và tính lại các chỉ số.